In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/Px_interface


In [2]:
%load_ext autoreload
%autoreload 2

import os, re, gc, ctypes, json, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date
from rdkit import Chem
import yaml
import joblib
from tqdm import tqdm
from types import SimpleNamespace
tqdm.pandas()

# local modules
import python.functions as fn
import python.Px_interface as px
from get_library import get_df   # CDD Vault collection export

In [3]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
## `config.RAW_PROTEOMICS_PATH` and bare `RAW_PROTEOMICS_PATH` work.
import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')

> loaded 41 params from config/config.yaml


## 0. Imports

In [4]:
%%time
## latest library straight from CDD Vault (collections AJ/AK), compound name + smiles only
if CHEMLIB_OVERWRITE:
    serac_df = (get_df(vault=7108, collections=['AK', 'AJ'], columns=['name', 'smiles','Px_repetition(yes/no)','Px_validated_WT(yes/no)','Px_Ligase_dependent(yes/no)',
                                                                      'Px_NameLigase_dependent','Px_Target_info','Px_Target_interest' ])
                .rename(columns={'name': 'compound'}))
    serac_df.to_csv(CHEMLIB_PATH,sep=',',index=False)
else:
    serac_df = pd.read_csv(CHEMLIB_PATH)
serac_df = serac_df.drop_duplicates()
serac_df['Px_validated_WT(yes/no)'] = serac_df['Px_validated_WT(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]
serac_df['Px_Ligase_dependent(yes/no)'] = serac_df['Px_Ligase_dependent(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]
serac_df['Px_repetition(yes/no)'] = serac_df['Px_repetition(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]

CPU times: user 98.3 ms, sys: 37 ms, total: 135 ms
Wall time: 138 ms


In [5]:
## get compounds that went to validation
val_cmps = list(set(serac_df[serac_df['Px_repetition(yes/no)']==1]['compound']))


In [6]:
## imports:
measure = pd.read_parquet(GTLOCAL+'Px_MEASURE.parquet') # 
mscore = pd.read_parquet(GTLOCAL+'Px_MSCORE.parquet') # the MS score of genes (one gene, many plates)
report = pd.read_parquet(GTLOCAL+'Px_REPORT.parquet') # the compound - plate
assoc  = pd.read_parquet(config.OT_CACHE).groupby('target_symbol')['overall_score'].max() # open target scores


#### 20260722 Wide validation excercise for Px

In [7]:
report['activity'].unique()

<ArrowStringArray>
['Medium (11-25)', 'Low (2-10)', 'Silent', 'Single (1)', 'High (>25)']
Length: 5, dtype: str

In [28]:
## reconcile Px interface selection: MS score > 10 & association > 0.6
## association = OpenTargets overall_score per gene (NOT mscore['association_score']).
DROP_PLATES   = ['Plate12', 'Plate15', 'Plate23']   # noisy plates the interface drops
ACTIVITY_BINS = ['Single (1)','Low (2-10)','Medium (11-25)'] # , 'Low (2-10)','Medium (11-25)'
ASSOC_CTF, MS_CTF = 0.6, 5
USE_MAX_MS = False   # True: gene passes if its BEST plate clears MS_CTF (interface); False: judge each (gene,plate)

## passing genes (max) or gene-plates (as-is): high OpenTargets association AND high ms_score
hi_assoc = assoc[assoc > ASSOC_CTF].index
if USE_MAX_MS:
    ms = mscore.groupby('genes')['ms_score'].max()
    passing, on = pd.DataFrame({'genes': ms.index[(ms > MS_CTF) & ms.index.isin(hi_assoc)]}), ['genes']
else:
    passing, on = mscore.loc[(mscore['ms_score'] > MS_CTF) & mscore['genes'].isin(hi_assoc), ['genes', 'plate']].drop_duplicates(), ['genes', 'plate']

## significant-down hits, joined to their compound, restricted to the passing genes / gene-plates
hits = measure.query("significant == 1 and logfc < 0 and plate not in @DROP_PLATES")
hits = hits.merge(report[['uniquecontrast', 'compound', 'activity']].drop_duplicates('uniquecontrast'), how='left')
sel  = hits[hits['compound'].isin(serac_df['compound']) & hits['activity'].isin(ACTIVITY_BINS)].merge(passing, on=on)

# assign compound # from SRB-XXXXXXX
sel['compound_no'] = sel['compound'].str.replace('SRB-','').astype(int)

print(f'genes: {sel["genes"].nunique():,}  |  compounds: {sel["compound"].nunique():,}')

final_number = len(sel[ (~sel['compound'].isin(val_cmps)) & (sel['compound_no'] >= 6400)  ]['compound'].unique())
print('> remaining compounds',final_number)

genes: 1,088  |  compounds: 1,839
> remaining compounds 784


In [27]:
## making the deliverable table:
## one number per (MS-score ctf, Association ctf, activity group): distinct NEW compounds
## (not in val_cmps, compound_no >= MIN_COMPOUND_NO) that are significant-down hits on a
## high-MS/high-association gene. Reproduces the reconcile-cell 87 (Single, MS 5, assoc 0.6).
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.oxml.ns import qn
from pptx.oxml import parse_xml

MS_CTFS    = [5, 10, 20]
ASSOC_CTFS = [0.6, 0.7]
GROUPS     = [('Single',              ['Single (1)']),
              ('Single+Low',          ['Single (1)', 'Low (2-10)']),
              ('Single+low+medium',   ['Single (1)', 'Low (2-10)', 'Medium (11-25)'])]
DROP_PLATES     = ['Plate12', 'Plate15', 'Plate23']
USE_MAX_MS      = False   # False = judge each (gene,plate); matches the reconcile 87
MIN_COMPOUND_NO = 6400
NEW_COMPOUNDS   = True    # True: compound_no >= MIN_COMPOUND_NO (new set); False: < it (legacy)
_tag = f'ge{MIN_COMPOUND_NO}' if NEW_COMPOUNDS else f'lt{MIN_COMPOUND_NO}'
OUT = f'/mnt/c/Users/gtamo/Desktop/GT/Claude_ppt/{date.today():%Y%m%d}_Px_selection_deliverable_{_tag}.pptx'

## base hits (heavy join done once): significant-down, library, NEW compounds only
base = measure.query("significant == 1 and logfc < 0 and plate not in @DROP_PLATES")
base = base.merge(report[['uniquecontrast', 'compound', 'activity']].drop_duplicates('uniquecontrast'), how='left')
base = base[base['compound'].isin(serac_df['compound']) & ~base['compound'].isin(val_cmps)]
base['compound_no'] = base['compound'].str.replace('SRB-', '').astype(int)
base = base[base['compound_no'] >= MIN_COMPOUND_NO] if NEW_COMPOUNDS else base[base['compound_no'] < MIN_COMPOUND_NO]
ms_max = mscore.groupby('genes')['ms_score'].max()

def count(ms_ctf, assoc_ctf, bins):
    hi = assoc[assoc > assoc_ctf].index
    if USE_MAX_MS:
        passing, on = pd.DataFrame({'genes': ms_max.index[(ms_max > ms_ctf) & ms_max.index.isin(hi)]}), ['genes']
    else:
        passing, on = mscore.loc[(mscore['ms_score'] > ms_ctf) & mscore['genes'].isin(hi), ['genes', 'plate']].drop_duplicates(), ['genes', 'plate']
    return base[base['activity'].isin(bins)].merge(passing, on=on)['compound'].nunique()

cols = [(g, a) for g, _ in GROUPS for a in ASSOC_CTFS]
grid = pd.DataFrame({(g, a): {m: count(m, a, dict(GROUPS)[g]) for m in MS_CTFS} for g, a in cols})
print(grid)

## ---- pptx ----
BLUE, PURPLE, ORANGE = RGBColor(0x2E, 0x77, 0xB5), RGBColor(0x93, 0x28, 0x8E), RGBColor(0xE8, 0x71, 0x2B)
GREY, WHITE, BLACK = RGBColor(0xE9, 0xEC, 0xEF), RGBColor(0xFF, 0xFF, 0xFF), RGBColor(0x00, 0x00, 0x00)
THICK, THIN = 28575, 12700   # EMU (2.25pt / 1pt)

def set_edges(cell, spec):
    tcPr = cell._tc.get_or_add_tcPr()
    for side in ('L', 'R', 'T', 'B'):                 # schema order: lnL, lnR, lnT, lnB
        tag = qn('a:ln' + side)
        e = tcPr.find(tag)
        if e is not None: tcPr.remove(e)
        w, col = spec.get(side, (THIN, 'FFFFFF'))
        tcPr.append(parse_xml(
            f'<a:ln{side} xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" '
            f'w="{w}" cap="flat"><a:solidFill><a:srgbClr val="{col}"/></a:solidFill></a:ln{side}>'))

def style(cell, text, fill, font_col, bold=True, size=18):
    cell.fill.solid(); cell.fill.fore_color.rgb = fill
    cell.vertical_anchor = MSO_ANCHOR.MIDDLE
    cell.margin_top = cell.margin_bottom = Pt(1)
    p = cell.text_frame.paragraphs[0]; p.alignment = PP_ALIGN.CENTER
    r = p.add_run(); r.text = text
    r.font.bold = bold; r.font.size = Pt(size); r.font.color.rgb = font_col

prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.33), Inches(7.5)
slide = prs.slides.add_slide(prs.slide_layouts[6])
nrow, ncol = 2 + len(MS_CTFS), 1 + len(cols)
gt = slide.shapes.add_table(nrow, ncol, Inches(1.7), Inches(1.1), Inches(10.6), Inches(3.6)).table
gt.first_row = gt.horz_banding = False
gt.columns[0].width = Inches(1.0)
for c in range(1, ncol): gt.columns[c].width = Emu(int((Inches(10.6) - Inches(1.0)) / len(cols)))
gt.rows[0].height = Inches(0.7)
for r in range(1, nrow): gt.rows[r].height = Inches(0.6)

## header row: one merged coloured cell per group, spanning its two assoc columns
style(gt.cell(0, 0), '', WHITE, BLACK)
for i, (gname, _) in enumerate(GROUPS):
    c0 = 1 + 2 * i
    gt.cell(0, c0).merge(gt.cell(0, c0 + 1))
    style(gt.cell(0, c0), gname, [BLUE, PURPLE, ORANGE][i], WHITE, size=20)

## data rows + left MS label
for ri, m in enumerate(MS_CTFS, start=1):
    style(gt.cell(ri, 0), str(m), WHITE, BLACK, size=18)
    for ci, (g, a) in enumerate(cols, start=1):
        style(gt.cell(ri, ci), str(int(grid.loc[m, (g, a)])), GREY, BLACK, bold=False, size=18)

## footer row: association cutoff ticks
style(gt.cell(nrow - 1, 0), '', WHITE, BLACK)
for ci, (g, a) in enumerate(cols, start=1):
    style(gt.cell(nrow - 1, ci), str(a), WHITE, BLACK, bold=False, size=16)

## borders: thin white grid inside, thick black around table + between the 3 groups
for r in range(nrow):
    for c in range(ncol):
        sp = {}
        if c == 0 or c == ncol - 1: sp['R' if c == 0 else 'L'] = (THICK, '000000')
        if c in (1, 3, 5): sp['L'] = (THICK, '000000')     # start of each group / MS-label divider
        if c in (2, 4, 6): sp['R'] = (THICK, '000000')     # end of each group
        if r in (0, 1, nrow - 1): sp['T'] = (THICK, '000000')
        if r in (0, nrow - 2, nrow - 1): sp['B'] = (THICK, '000000')
        set_edges(gt.cell(r, c), sp)

## axis titles
tb = slide.shapes.add_textbox(Inches(0.3), Inches(2.0), Inches(1.6), Inches(1.0))
tb.rotation = 270
pr = tb.text_frame.paragraphs[0]; pr.alignment = PP_ALIGN.CENTER
rr = pr.add_run(); rr.text = 'MS Scores ctf'; rr.font.bold = True; rr.font.size = Pt(20); rr.font.color.rgb = BLUE
xb = slide.shapes.add_textbox(Inches(1.7), Inches(4.9), Inches(10.6), Inches(0.6))
px = xb.text_frame.paragraphs[0]; px.alignment = PP_ALIGN.CENTER
rx = px.add_run(); rx.text = 'Association Score ctf'; rx.font.bold = True; rx.font.size = Pt(20); rx.font.color.rgb = BLUE

os.makedirs(os.path.dirname(OUT), exist_ok=True)
prs.save(OUT)
print('> saved', OUT)

   Single     Single+Low      Single+low+medium     
      0.6 0.7        0.6  0.7               0.6  0.7
5      87  63        680  522               784  621
10     39  31        380  279               472  361
20     15  14        128   93               182  136
> saved /mnt/c/Users/gtamo/Desktop/GT/Claude_ppt/20260722_Px_selection_deliverable.pptx
